In [1]:
%load_ext autoreload
%autoreload 2

# Importing libraries

All the sensors are represented as objects in the module `oceanicospy.observations`.

In [34]:
from oceanicospy.observations import pressure_sensors,weather_stations,AWAC,Buoy
from datetime import datetime, timedelta
import glob

# Pressure sensors: AQUALogger, RBR, BlueLogger

## Data directories and sampling configuration

For every sensor, we specify the directory where the data files are located and a sampling configuration dictionary that contains parameters such as anchoring depth, sensor height, sampling frequency, burst length, and optional start and end times for filtering the data.

In [3]:
measurement_pressure_sensors_paths = ['../data/observations/AQUAlogger/','../data/observations/RBR/','../data/observations/Bluelog/']

sampling_AQ = dict(anchoring_depth=1, sensor_height=0.2, sampling_freq=1,burst_length_s=2048,
                    start_time=datetime(2025,5,9,10,0,0),end_time=datetime(2025,5,19,18,0,0)-timedelta(seconds=1))
sampling_RBR = dict(anchoring_depth=1, sensor_height=0.2, sampling_freq=2, burst_length_s=7200,
                    start_time=datetime(2025,5,9,10,0,0),end_time=datetime(2025,5,19,18,0,0)-timedelta(seconds=0.5))
sampling_Bluelog = dict(anchoring_depth=1, sensor_height=0.2, sampling_freq=2, burst_length_s=4096, temperature=False,
                        start_time=datetime(2025,7,4,14,0,0), end_time=datetime(2025,7,10,12,0,0)-timedelta(seconds=1))

## Reading and storing data

In [4]:

sampling_data = [sampling_AQ,sampling_RBR,sampling_Bluelog]
metadata_list=['AQ','RBR','Bluelog']
dict_raw_measurements = dict()
dict_clean_measurements = dict() 

for idx,measurement_path in enumerate(measurement_pressure_sensors_paths):
    print(measurement_path)
    if 'AQ' in measurement_path:
        object_device = pressure_sensors.AQUAlogger(measurement_path, sampling_data[idx],filename='AQUAlogger_520PT5.csv')
    elif 'Bluelog' in measurement_path:
        object_device = pressure_sensors.Bluelog(measurement_path, sampling_data[idx])
    else:
        object_device = pressure_sensors.RBR(measurement_path,sampling_data[idx])
    raw_data = object_device.get_raw_records()
    clean_data = object_device.get_clean_records()
    dict_raw_measurements[metadata_list[idx]] = raw_data
    dict_clean_measurements[metadata_list[idx]] = clean_data

../data/observations/AQUAlogger/
../data/observations/RBR/
../data/observations/Bluelog/


A quick look at the data dictionary can show the available data for each sensor. Each sensor's data is retrieved as a pandas DataFrame, which allows for easy manipulation and analysis.

In [5]:
dict_clean_measurements['AQ']

,pressure[bar],depth[m],depth_aux[m],burstId,eta[m]
date,,,,,
2025-05-09 10:00:00,1.245722,3.136509,2.307327,1,0.052596
2025-05-09 10:00:01,1.244709,3.124966,2.297273,1,0.041052
2025-05-09 10:00:02,1.237181,3.039040,2.222556,1,-0.044874
2025-05-09 10:00:03,1.243695,3.113418,2.287209,1,0.029503
2025-05-09 10:00:04,1.253249,3.222086,2.382034,1,0.138170
...,...,...,...,...,...
2025-05-19 16:34:03,1.239498,3.065513,2.245553,124,-0.003209
2025-05-19 16:34:04,1.242248,3.096910,2.272847,124,0.028177
2025-05-19 16:34:05,1.237905,3.047316,2.229742,124,-0.021427


In [6]:
dict_clean_measurements['RBR']

,pressure[bar],depth[m],burstId,eta[m]
date,,,,
2025-05-09 10:00:00.000,1.241829,2.267149,1,0.117985
2025-05-09 10:00:00.500,1.239702,2.246051,1,0.096886
2025-05-09 10:00:01.000,1.235314,2.202530,1,0.053365
2025-05-09 10:00:01.500,1.233978,2.189276,1,0.040110
2025-05-09 10:00:02.000,1.234466,2.194119,1,0.044954
...,...,...,...,...
2025-05-19 17:59:57.500,1.228489,2.134836,248,-0.044336
2025-05-19 17:59:58.000,1.228343,2.133389,248,-0.045786
2025-05-19 17:59:58.500,1.226372,2.113838,248,-0.065340


In [7]:
dict_clean_measurements['Bluelog']

,pressure[bar],temperature[c],depth[m],burstId,eta[m]
date,,,,,
2025-07-04 14:00:00.000,1.0615,27.80,0.478890,1,-0.000014
2025-07-04 14:00:00.500,1.0616,27.80,0.479883,1,0.000980
2025-07-04 14:00:01.000,1.0617,27.80,0.480875,1,0.001974
2025-07-04 14:00:01.500,1.0617,27.80,0.480875,1,0.001976
2025-07-04 14:00:02.000,1.0615,27.80,0.478890,1,-0.000008
...,...,...,...,...,...
2025-07-10 11:34:05.500,2.1055,27.65,10.840782,142,0.014486
2025-07-10 11:34:06.000,2.1045,27.65,10.830857,142,0.004558
2025-07-10 11:34:06.500,2.1037,27.65,10.822917,142,-0.003385


# ADCP sensors: AWAC

Support for ADCP sensors, such as the AWAC, is also included in the module. The `AWAC` class is designed to handle the specific data formats and requirements of ADCP measurements, allowing users to easily load and process wave records from AWAC deployments.

Similar to the pressure sensors, the `AWAC` class also requires a directory path and sampling configuration for initialization. 

In [10]:
measurement_AWAC_paths = ['../data/observations/AWAC/']

sampling_AWAC=dict(anchoring_depth=1,sensor_height=0.2,sampling_freq=1,burst_length_s=2048,temperature=False,
                            start_time=datetime(2023,8,18,19,0,0),end_time=datetime(2023,9,1,15,0,0)-timedelta(seconds=1))


In [11]:
help(AWAC)

Help on class AWAC in module oceanicospy.observations.awac:

class AWAC(builtins.object)
 |  AWAC(directory_path, sampling_data)
 |  
 |  A class to handle reading and processing the data files recorded by an ADCP AWAC (Nortek S.A). 
 |  
 |  Parameters
 |  ----------
 |  directory_path : str
 |      Path to the directory containing the .hdr and .wad files.
 |  sampling_data : dict
 |      Dictionary containing the information about the device installation
 |  
 |  Notes
 |  -----
 |  
 |  - 04-Jan-2018 : Origination - Daniel Peláez
 |  - 01-Sep-2023 : Migration to Python - Alejandro Henao
 |  - 10-Dec-2024 : Class implementation - Franklin Ayala
 |  
 |  Methods defined here:
 |  
 |  __init__(self, directory_path, sampling_data)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  get_clean_currents_records(self, compute_speed_dir=True)
 |      Processes the raw current data by reading the x and y components, setting the index to a date range, and optionall

In [12]:
AWAC_obj = AWAC(measurement_AWAC_paths[0],sampling_AWAC)

The method `get_raw_wave_records` can be used to retrieve the raw wave records, with an option to specify whether to read from a single WAD file or multiple files in the directory.

In [25]:
df_wave_single_wad = AWAC_obj.get_raw_wave_records(from_single_wad=True)
df_wave_combined_wad = AWAC_obj.get_clean_wave_records(from_single_wad=False)

In [ ]:
df_wave_single_wad['3 Pressure (dbar)']

0         0.000
1         0.000
2         0.000
3         0.000
4         0.000
          ...  
679931    0.000
679932    0.001
679933    0.000
679934    0.002
679935    0.000
Name: 3 Pressure (dbar), Length: 679936, dtype: float64

In [27]:
df_wave_combined_wad.columns

Index(['pressure[bar]', 'burstId', 'velocitybeam1[m/s]', 'velocitybeam2[m/s]',
       'velocitybeam3[m/s]', 'velocitybeam4[m/s]'],
      dtype='object')

In [30]:
df_wave_combined_wad['pressure[bar]']

2023-08-18 19:00:00.000    0.0000
2023-08-18 19:00:00.500    0.0000
2023-08-18 19:00:01.000    0.0000
2023-08-18 19:00:01.500    0.0000
2023-08-18 19:00:02.000    0.0000
                            ...  
2023-09-01 14:17:01.500    0.0000
2023-09-01 14:17:02.000    0.0001
2023-09-01 14:17:02.500    0.0000
2023-09-01 14:17:03.000    0.0002
2023-09-01 14:17:03.500    0.0000
Name: pressure[bar], Length: 679936, dtype: float64

It is quite weird that the single wad file does not start at 00:00 (minutes/seconds)

In [13]:
x_currents,y_currents = AWAC_obj.get_raw_currents_records()

In [17]:
AWAC_obj.get_clean_currents_records()

AttributeError: 'AWAC' object has no attribute 'x_component'

# Weather stations

Currently, there is support for three different weather station models: Davis Vantage Pro, WeatherSens and Rainwise

In [13]:
Davis_weather_station = weather_stations.DavisVantagePro('../data/observations/WeatherStations/DavisVantagePro.txt')
WeatherSens_weather_station = weather_stations.WeatherSens('../data/observations/WeatherStations/WeatherSens.xlsx')
Rainwise_weather_station = weather_stations.Rainwise('../data/observations/WeatherStations/Rainwise.csv')

object_list = [Davis_weather_station,WeatherSens_weather_station,Rainwise_weather_station]

In [15]:
dict_raw_measurements = dict()
dict_clean_measurements = dict()

for ws in object_list:
    raw_data = ws.get_raw_records()
    clean_data = ws.get_clean_records()
    dict_raw_measurements[ws.__class__.__name__] = raw_data
    dict_clean_measurements[ws.__class__.__name__] = clean_data

/Users/franklinayala/Documents/projects/oceanicospy/oceanicospy/observations/weather_stations/davis.py:50: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace('---', np.nan, inplace=True)
/Users/franklinayala/venvs/oceanicospy-dev-ffayalac/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/franklinayala/venvs/oceanicospy-dev-ffayalac/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [23]:
df = dict_clean_measurements['DavisVantagePro']

# Spotter buoy

In [39]:
csv_files_spotter = glob.glob('../data/observations/Buoy/*.csv')
dict_spotter_sources = dict()
for csv_file in csv_files_spotter:
    spotter_object = Buoy(csv_file)
    dict_spotter_sources[csv_file.split('/')[-1].split('.')[0]] = spotter_object.get_raw_records()

In [41]:
dict_spotter_sources['spotter_data_aqualink']

,timestamp,wind_speed_gfs,wind_direction_gfs,temp_alert_noaa,temp_weekly_alert_noaa,dhw_noaa,satellite_temperature_noaa,sst_anomaly_noaa,top_temperature_spotter,bottom_temperature_spotter,significant_wave_height_spotter,wave_mean_period_spotter,wave_mean_direction_spotter,wind_speed_spotter,wind_direction_spotter,significant_wave_height_sofar_model,wave_mean_period_sofar_model,wave_mean_direction_sofar_model
date,,,,,,,,,,,,,,,,,,
2022-05-01 01:00:00,2022-05-01T06:00:00.000Z,5.116343,24.501352,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.484210,5.385854,70.847834
2022-05-01 07:00:00,2022-05-01T12:00:00.000Z,6.465680,27.656962,0.0,0.0,0.000000,27.570000,0.060666,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.387397,5.392312,69.502012
2022-05-01 13:00:00,2022-05-01T18:00:00.000Z,5.919429,40.296021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.339071,5.306848,68.041002
2022-05-01 19:00:00,2022-05-02T00:00:00.000Z,7.152353,29.802939,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.361218,5.088913,62.355534
2022-05-02 01:00:00,2022-05-02T06:00:00.000Z,6.601567,40.623034,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.426305,5.052871,59.526641
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-08-27 07:00:00,2023-08-27T12:00:00.000Z,NaN,NaN,2.0,2.0,3.359647,29.899956,1.791569,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-08-28 07:00:00,2023-08-28T12:00:00.000Z,NaN,NaN,2.0,2.0,3.399530,30.049994,1.930962,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-08-29 07:00:00,2023-08-29T12:00:00.000Z,NaN,NaN,2.0,2.0,3.439877,29.979915,1.850237,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [42]:
dict_spotter_sources['spotter_data_sofar']

,Battery Voltage (V),Power (W),Humidity (%rel),Epoch Time,Significant Wave Height (m),Peak Period (s),Mean Period (s),Peak Direction (deg),Peak Directional Spread (deg),Mean Direction (deg),...,Partition0 Mean Direction (deg),Partition0 Mean Directional Spread (deg),Partition1 Start Frequency (hz),Partition1 End Frequency (hz),Partition1 Significant Wave Height (m),Partition1 Mean Period (s),Partition1 Mean Direction (deg),Partition1 Mean Directional Spread (deg),Mean Barometric Pressure (hPa),Processing Source
date,,,,,,,,,,,,,,,,,,,,,
2025-05-01 00:40:00,4.01,-0.14,81.6,1746078000,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,embedded
2025-05-01 01:10:00,4.01,-0.14,81.6,1746079800,0.634,6.827,3.221,81.378,69.047,58.304,...,-,-,-,-,-,-,-,-,1011.7,embedded
2025-05-01 02:40:00,4.01,-0.14,82.4,1746085200,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,embedded
2025-05-01 03:10:00,4.01,-0.14,82.4,1746087000,0.628,6.827,3.291,73.11,67.33,57.726,...,-,-,-,-,-,-,-,-,1010.6,embedded
2025-05-01 04:40:00,3.99,-0.14,83.2,1746092400,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,embedded
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-05-31 19:10:00,4.04,-0.08,75.2,1748736600,0.582,8.533,3.373,111.922,73.695,55.895,...,-,-,-,-,-,-,-,-,1009.5,embedded
2025-05-31 20:40:00,4.04,-0.15,80.8,1748742000,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,embedded
2025-05-31 21:10:00,4.04,-0.15,80.8,1748743800,0.576,7.314,3.264,122.074,69.724,59.15,...,-,-,-,-,-,-,-,-,1010.9,embedded


# CTD sensors

# Hobo sensor

# Mantaray